# Notebook 03 — Real Embeddings (384 dimensions, free & local)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

So far: hand-built 4-D vectors — great for intuition, useless for real text. Now we use a
real model, **all-MiniLM-L6-v2** from HuggingFace, free and running on your own machine. It
turns each sentence into a **384-dimensional** vector. Every tool from Notebooks 01–02
(cosine similarity, nearest neighbours, PCA) works unchanged. Only the number of directions
grows: 4 → 384.

**Cost:** zero. No account, no API key. The model runs locally.

In [ ]:
# Install the libraries this notebook uses. -q keeps the output short.
# sentence-transformers brings in the MiniLM model; the others handle math and the plot.
%pip install -q numpy matplotlib scikit-learn sentence-transformers
print("Ready.")

## Step 1 — Load the model

> HuggingFace is like a public library of trained models: you borrow one for free, no
> sign-up. The first run downloads MiniLM (about 80 MB) once, then loads instantly after.

The cell below imports our tools and loads the model into a variable called `model`. When it
finishes you will see one line: `Model loaded...`. On the first run a short download bar may
appear above it. Nothing else happens yet; we have only picked up the tool.

In [ ]:
import numpy as np                              # arrays and fast math
import matplotlib.pyplot as plt                  # the plot at the end
from sklearn.decomposition import PCA            # squashes 384-D down to 2-D for the plot
from sentence_transformers import SentenceTransformer  # the wrapper that loads MiniLM

# Download (first time) then load the model by name. Returns an object we call to embed text.
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. It turns any text into a 384-number vector.")

## Step 2 — Twelve sentences

Four sentences each from three obvious groups: **animals**, **vehicles**, **food**. The
model never sees these labels; we will check whether it groups them anyway.

The cell only stores the text and the labels, so it prints one line confirming
**12 sentences, 3 groups**. The `categories` and `colour_map` we set up here are used later
to colour the plot.

In [ ]:
sentences = [
    # Animals
    "The dog wagged its tail when its owner came home.",
    "A kitten chased a ball of yarn across the floor.",
    "Lions live in prides on the African savanna.",
    "The parrot mimicked every word the children said.",
    # Vehicles
    "I drove the car to the grocery store this morning.",
    "The new electric bicycle has a range of sixty miles.",
    "Trucks deliver packages to our neighbourhood every day.",
    "The airplane landed smoothly despite the strong winds.",
    # Food
    "She baked sourdough bread for the first time on Sunday.",
    "The pizza was hot and covered in melted mozzarella.",
    "I ordered sushi for lunch at the new Japanese restaurant.",
    "He grilled steak and roasted vegetables for dinner.",
]
# One label per sentence, same order: first four animal, next four vehicle, last four food.
categories = ["animal"] * 4 + ["vehicle"] * 4 + ["food"] * 4
# A colour per label, used only when we draw the plot at the end.
colour_map = {"animal": "tab:orange", "vehicle": "tab:blue", "food": "tab:red"}

# set(categories) drops duplicates, so this counts the distinct groups (3).
print(f"{len(sentences)} sentences, {len(set(categories))} groups.")

## Step 3 — Embed all twelve at once

`model.encode(list_of_sentences)` runs every sentence through the model and hands back one
vector per sentence, packed into a NumPy array of shape **(12, 384)**: twelve sentences, each
384 numbers. Compare with Notebook 01: ten words, each 4 numbers. Same idea, more dials.

When you run the next cell, the shape prints as **(12, 384)**, the type is `float32`, and you
see the first five numbers of sentence 0 (small positive and negative decimals). The model is
deterministic on CPU, so you should see the same numbers each run, give or take a wobble in
the last digit on a different machine.

In [ ]:
# One call embeds all twelve sentences. Result is a NumPy array, one row per sentence.
embeddings = model.encode(sentences)

print(f"Shape   : {embeddings.shape}   (sentences, directions)")  # (12, 384)
print(f"Type    : {embeddings.dtype}")                            # float32: 384 decimals per row
print(f"First 5 of sentence 0: {embeddings[0, :5]}")              # a peek at the raw numbers
print("\nEmbedded. Each sentence is now 384 numbers.")

### What to notice
- **(12, 384)** — twelve sentences, each a 384-number vector.
- The numbers look like noise to us, but they encode meaning. We read them through
  *similarity scores* and *plots*, never directly.
- Twelve sentences is tiny. Picture a million: 1,000,000 × 384 numbers. Storing and
  searching that is the job of a vector store — Notebook 04.

## Step 4 — Cosine similarity, two ways

Cosine similarity scores how close two vectors point: -1 (opposite), 0 (unrelated), 1 (same
direction). We score it two ways. First with **our own** function from Notebook 01 (proof the
idea did not change at 384 dimensions), then the fast NumPy way that does all pairs at once.

The next cell defines the helper and prints `our cosine(dog, kitten)`. Both are animal
sentences, but worded very differently, so expect a modest score (about 0.25 here), nowhere
near 1.0.

In [ ]:
# Activity: our own cosine from Notebook 01 (works on a vector of any length).
import math

def dot_product(a, b):
    # Multiply matching positions and add them up: the heart of cosine.
    return sum(x * y for x, y in zip(a, b))

def magnitude(v):
    # Length of the vector: square every number, sum, take the square root.
    return math.sqrt(sum(x * x for x in v))

def cosine_similarity(a, b):
    # Dot product divided by both lengths. This cancels size and leaves direction alone.
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

dog, kitten = embeddings[0], embeddings[1]  # rows 0 and 1: the dog and kitten sentences
# Two animal sentences with no shared words, so this lands modestly above zero, not near 1.0.
print(f"our cosine(dog, kitten) = {cosine_similarity(dog, kitten):.3f}")

## The opening example: no shared words, still similar

Before the twelve sentences, here is the trio from the lesson. The two dog/puppy sentences
share **no words**, yet they score far higher than either does against the stock-market
sentence. Meaning, not spelling.

Expect about **0.61** for dogs vs puppies and about **0.03** for dogs vs the stock market.
The gap is the point: close meaning scores high, unrelated meaning scores near zero.

In [ ]:
intro = ["I love dogs", "Puppies are wonderful", "The stock market fell sharply"]
iv = model.encode(intro)  # embed all three; iv[0], iv[1], iv[2] are the three vectors
# Same idea, different words: scores high (about 0.61).
print("dogs vs puppies     :", round(cosine_similarity(iv[0], iv[1]), 3))
# Unrelated topic: scores near zero (about 0.03).
print("dogs vs stock market:", round(cosine_similarity(iv[0], iv[2]), 3))

Now the fast library way. We normalise every row to length 1, then a single matrix
multiply gives all 12 by 12 pairs at once. We check it lands on the same `dog, kitten` number
our own function gave, so you can trust the shortcut.

Expect three lines: the dog/kitten score matching the earlier 0.249, `cosine(dog, dog) = 1.000`
(anything compared with itself points the same way), and a low score for dog vs the sourdough
sentence (different topics).

In [ ]:
# Activity: the library way (normalize every row, then one matrix multiply).
# Divide each row by its own length so every vector has length 1. keepdims keeps the shapes lined up.
norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
# With unit-length rows, row times column is exactly cosine. One multiply fills the whole 12x12 grid.
sim_matrix = norm @ norm.T

# Same dog/kitten pair as before: confirms the shortcut matches our handwritten cosine.
print(f"library cosine(dog, kitten)  = {sim_matrix[0, 1]:.3f}   (same number)")
# A vector with itself always scores 1.0 (the diagonal of the grid).
print(f"cosine(dog, dog)             = {sim_matrix[0, 0]:.3f}   (a thing with itself = 1.0)")
# Sentence 0 (dog) against sentence 8 (sourdough): different topics, so lower.
print(f"cosine(dog, sourdough bread) = {sim_matrix[0, 8]:.3f}   (different topics = lower)")

## Words vs sentences, and how context steers a word

A single word is just a very short text, so the model embeds it the same way it embeds a
sentence, and you can compare a word against a sentence with the same cosine. The next two
cells show this, then run the famous "bank" test: two sentences that BOTH contain the word
"bank", one about a river and one about money. We probe each with a water idea and a money
idea and watch the surrounding words steer the meaning.

In the first cell, expect `puppy vs dog` to score high (about 0.80) and `puppy vs car` much
lower (about 0.46). A bare word against a full sentence scores lower again (about 0.47 for
dog vs the dog sentence, about 0.10 for dog vs the pizza sentence), because the sentence
carries extra meaning the lone word does not.

In [ ]:
# Activity: compare a word to a word, and a word to a sentence.
w_puppy = model.encode("puppy")   # one word in, one 384-number vector out
w_dog = model.encode("dog")
w_car = model.encode("car")

print("word vs word:")
print("  puppy vs dog:", round(cosine_similarity(w_puppy, w_dog), 3))   # ~0.804 close
print("  puppy vs car:", round(cosine_similarity(w_puppy, w_car), 3))   # ~0.464 far

print("word vs sentence:")
# Compare the bare word "dog" against full sentences (rows 0 and 9 of embeddings).
print("  dog vs dog sentence  :", round(cosine_similarity(w_dog, embeddings[0]), 3))  # ~0.470
print("  dog vs pizza sentence:", round(cosine_similarity(w_dog, embeddings[9]), 3))  # ~0.10

In [ ]:
# Activity: the "bank" test — context steers the same word two different ways.
A = model.encode("We sat on the river bank and watched the water.")   # here "bank" = riverside
B = model.encode("I deposited my salary at the bank this morning.")   # here "bank" = money
water = model.encode("a river with flowing water")  # a probe that leans toward the water meaning
money = model.encode("money and finance")           # a probe that leans toward the money meaning

# Both sentences contain "bank", yet they are not close: the rest of the words pull them apart.
print("A vs B (both contain 'bank'):", round(cosine_similarity(A, B), 3))   # ~0.281 NOT close!
# The river sentence sits closer to the water probe than to the money probe.
print("A vs water probe:", round(cosine_similarity(A, water), 3))           # ~0.531 high
print("B vs water probe:", round(cosine_similarity(B, water), 3))           # ~0.098 low
# The salary sentence flips it: closer to the money probe than to the water probe.
print("A vs money probe:", round(cosine_similarity(A, money), 3))           # ~0.190 low
print("B vs money probe:", round(cosine_similarity(B, money), 3))           # ~0.301 higher

## Know your model: dimension, max length, size

Three facts to know about any embedding model. For ours: dimension **384**, max sequence
length **256 tokens** (a token is a word or word-piece; "wagged" reads as `wa` + `gged`, and
256 tokens is roughly 190 to 200 words), size about **80 MB**. The sharp edge: text past the
limit is **silently cut off**.

The next cell prints these facts straight from the model (256, 384, and the two pieces
"wagged" splits into). The cell after that proves the truncation: the same clue read near the
front of a long document scores higher than the same clue buried past the 256-token cut,
where the model never sees it.

In [ ]:
# Activity: read the model's own fact sheet.
print("max sequence length:", model.max_seq_length, "tokens")              # 256: the input limit
print("dimension:", model.get_sentence_embedding_dimension())              # 384: numbers per vector
# Show how the tokenizer splits one word into word-pieces (this is what "256 tokens" counts).
print("'wagged' becomes the tokens:", model.tokenizer.tokenize("wagged"))

In [ ]:
# Activity: the truncation trap — text past 256 tokens is silently ignored.
# Repeat a dull sentence 40 times to build text that overflows the 256-token limit.
filler = "The committee reviewed the quarterly schedule and noted the agenda. " * 40

probe = model.encode("pizza and italian food")  # what we will search the documents for
# Put the pizza clue AFTER the filler, so it falls past the cut and the model never reads it.
at_end   = model.encode(filler + " The secret topic of this document is pizza.")
# Put the pizza clue BEFORE the filler, so it sits inside the first 256 tokens and is read.
at_front = model.encode("The secret topic of this document is pizza. " + filler)

# Clue past the limit: score stays low, as if the pizza line were never there.
print("pizza at the END  :", round(cosine_similarity(probe, at_end), 3))    # ~0.102 never read
# Clue inside the limit: score rises, though the filler dilutes it.
print("pizza at the FRONT:", round(cosine_similarity(probe, at_front), 3))  # ~0.231 read, diluted
# (You may see a tokenizer warning about the text being too long — that's the point!)

## Step 5 — Predict, then verify: three searches

For each query, guess which of the twelve sentences is **closest** in meaning before you run
it. The queries share almost no words with their best matches, so meaning is what counts here,
not shared words.

- Q1: "a pet that climbs trees and purrs"
- Q2: "how do I travel between cities?"
- Q3: "what should I cook for guests tonight?"

The next cell prints, for each query, the closest and farthest sentence with its score.
Expect the closest match to be an animal, then a vehicle, then a food sentence, with top
scores in the rough range 0.2 to 0.45, well below 1.0. Some scores for unrelated sentences
even dip slightly below zero.

In [ ]:
queries = [
    "a pet that climbs trees and purrs",
    "how do I travel between cities?",
    "what should I cook for guests tonight?",
]
query_vecs = model.encode(queries)  # embed all three queries; one vector each

for q, qv in zip(queries, query_vecs):
    # Score this query against every one of the twelve sentence vectors.
    sims = np.array([cosine_similarity(qv, ev) for ev in embeddings])
    top = int(np.argmax(sims))     # index of the highest score: the best match
    bottom = int(np.argmin(sims))  # index of the lowest score: the worst match
    print(f"\nQuery: {q!r}")
    print(f"  CLOSEST  [{sims[top]:.3f}]  {sentences[top]!r}")
    print(f"  FARTHEST [{sims[bottom]:.3f}]  {sentences[bottom]!r}")

### What to notice
- **It understands paraphrase.** "a pet that climbs trees and purrs" finds an *animal*
  sentence — even though it shares no words like "climb" or "purr" with it. (For this small
  set the top animal match is the dog sentence, not the kitten — close calls happen. The
  model deals in shades of meaning, not certainties.)
- **The top score is not 1.0.** Real matches score low — often ~0.2–0.5 here. Don't expect
  near-1.0; that only happens for nearly identical text. What matters is *which* is highest.

## Step 6 — See the twelve sentences with PCA

Same PCA as Notebook 02, now squashing **384 down to 2** instead of 4 down to 2. More is lost
in the squash, so the picture is rougher, but the three groups should still cluster.

The next cell draws a scatter plot, each point labelled with the start of its sentence and
coloured by group, then prints how much of the spread the two axes capture. Expect a smallish
percentage (around 28 percent here, far below the 4-D case), because at 384 dimensions most
of the meaning lives in directions a flat picture cannot show.

In [ ]:
pca = PCA(n_components=2)              # set up PCA to keep the 2 strongest directions
pcs = pca.fit_transform(embeddings)   # squash each 384-number row down to an (x, y) pair
var = pca.explained_variance_ratio_   # fraction of the spread each of the 2 axes captures

plt.figure(figsize=(10, 6))
# Plot every sentence as a dot, coloured by its group, labelled with its first 25 characters.
for (x, y), cat, sent in zip(pcs, categories, sentences):
    plt.scatter(x, y, c=colour_map[cat], s=150, edgecolor="black")
    plt.annotate(sent[:25] + "…", (x, y), xytext=(6, 4),
                 textcoords="offset points", fontsize=9)
# Add empty points only so the legend shows one entry per group.
for cat, c in colour_map.items():
    plt.scatter([], [], c=c, s=150, edgecolor="black", label=cat)
plt.legend(loc="best")
plt.xlabel(f"PC1  ({var[0] * 100:.0f}% of spread)")   # how much PC1 captures
plt.ylabel(f"PC2  ({var[1] * 100:.0f}% of spread)")   # how much PC2 captures
plt.title("12 sentences in 384-D, flattened to 2-D")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Add the two fractions: the share of the spread the flat picture keeps.
print(f"PC1 + PC2 capture {sum(var) * 100:.0f}% of the spread "
      f"(much less than the 4-D case — most meaning is in the other directions).")

## Recap

- A real model turns each sentence into a 384-number vector with one `encode` call.
- Read an embedding's shape: first number = how many items, second = the vector size.
- Our hand-written cosine and the library's give the same answer — the idea never changed.
- The model matches *meaning*, not shared words; top cosines are well below 1.0.
- A PCA plot is a rough approximation of high-D similarity. When the plot and the cosine
  disagree, trust the cosine.

**Next (Notebook 04):** we found nearest sentences by comparing against all twelve by hand.
At a million sentences that breaks down. A **vector store** (ChromaDB) does the storing and
searching for us.

## Practice — Your Turn

Three short exercises to make the ideas stick. Read each task, make a guess where it asks for
one, then run the answer cell below it to check. Everything uses the `model` and the
`cosine_similarity` function already defined above, so nothing new needs installing.

A reminder on the numbers: these come from a real model, so cosine scores for related text
usually sit around 0.2 to 0.6, not near 1.0. Only nearly identical text scores close to 1.0.
Your numbers should land within about two decimals of the ones shown in the comments. A small
wobble in the last digit on a different machine is normal.

### Exercise 1 — Score your own sentences

Write three short sentences of your own. Make two of them about the same idea (worded
differently, sharing as few words as possible) and one about something unrelated. For
example: "I went hiking in the mountains.", "We hiked a steep trail all afternoon.", and "The
printer is out of ink."

Before you run anything, predict which pair will score higher. The two related sentences
should win, even when they share almost no words, because the model matches meaning. Both
scores will sit somewhere in the rough 0.0 to 0.6 band, well short of 1.0, and your numbers
should match ours to about two decimals if you use the same example sentences.

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
# Two sentences about the same idea (hiking), worded differently, plus one unrelated sentence.
related_a = "I went hiking in the mountains."        # idea: hiking outdoors
related_b = "We hiked a steep trail all afternoon."  # same idea, almost no shared words
unrelated = "The printer is out of ink."             # a different topic entirely

va = model.encode(related_a)  # turn the first sentence into a 384-number vector
vb = model.encode(related_b)  # the second sentence
vu = model.encode(unrelated)  # the unrelated sentence

# Cosine between the two hiking sentences: higher, because they mean the same thing.
print("related pair    (a vs b):", round(cosine_similarity(va, vb), 3))   # about 0.59
# Cosine between a hiking sentence and the printer sentence: near zero, different topics.
print("unrelated pair  (a vs u):", round(cosine_similarity(va, vu), 3))   # about -0.03
# The related pair should print the larger of the two numbers.
print("related pair scored higher:", cosine_similarity(va, vb) > cosine_similarity(va, vu))

### Exercise 2 — The vector length never changes

The model always returns 384 numbers, whatever you feed it. A single letter, one word, a long
sentence: all come back as a vector of length 384. The dimension is a property of the model,
not of the text. The text decides what the numbers are, the model decides how many there are.

Predict the length for each input below before you run it. They should all print 384.

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
# A mix of inputs: a single letter, one word, a full sentence, and a number written as text.
inputs = [
    "a",                                                  # one letter
    "elephant",                                           # one word
    "The quick brown fox jumps over the lazy dog repeatedly.",  # a full sentence
    "42",                                                 # a number as text
]

# Embed each input on its own and print the length of the vector that comes back.
for text in inputs:
    vec = model.encode(text)          # one input in, one 384-number vector out
    print(f"len = {len(vec):3d}   <- {text!r}")  # the length is 384 every time

# A quick all-in-one check: every length equals 384, so this prints True.
print("all 384:", all(len(model.encode(t)) == 384 for t in inputs))

### Exercise 3 — A context test on the word "spring"

The lesson ran the "bank" test. Here is the same idea on another word with two meanings:
"spring" can be a source of water or a coiled piece of metal. We build two sentences, one for
each meaning, then two short probes that stand for the two meanings, and check which probe
each sentence leans toward.

- Sentence 1: "The spring bubbled with fresh water." (the water meaning)
- Sentence 2: "The spring on the door snapped." (the metal meaning)
- Probe 1: "a source of flowing water"
- Probe 2: "a coiled metal part"

Predict first: the water sentence should sit closer to the water probe, and the door sentence
closer to the metal probe. The four cosines land in the rough 0.0 to 0.4 band, and for each
sentence the score for its own meaning should be the larger of its two.

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
# Two "spring" sentences, one per meaning. The surrounding words carry the real sense.
s_water = model.encode("The spring bubbled with fresh water.")  # spring = a water source
s_metal = model.encode("The spring on the door snapped.")       # spring = a coiled metal part

# Two short probes that stand for the two meanings.
p_water = model.encode("a source of flowing water")  # leans toward the water meaning
p_metal = model.encode("a coiled metal part")        # leans toward the metal meaning

# The water sentence against each probe: it should sit closer to the water probe.
print("water sentence vs water probe:", round(cosine_similarity(s_water, p_water), 3))  # about 0.40
print("water sentence vs metal probe:", round(cosine_similarity(s_water, p_metal), 3))  # about 0.23

# The door sentence against each probe: it should sit closer to the metal probe.
print("metal sentence vs water probe:", round(cosine_similarity(s_metal, p_water), 3))  # about 0.01
print("metal sentence vs metal probe:", round(cosine_similarity(s_metal, p_metal), 3))  # about 0.28

# Each sentence leans toward its own meaning, so both of these print True.
print("water sentence leans water:", cosine_similarity(s_water, p_water) > cosine_similarity(s_water, p_metal))
print("metal sentence leans metal:", cosine_similarity(s_metal, p_metal) > cosine_similarity(s_metal, p_water))